# UniTopRank Resolution

## Goal

Use UniTopRank to rank candidate locations for place names extracted from raw text.

UniTopRank resolves place names by first collecting candidate locations from gazetteers such as GeoNames and Photon, then ranking those candidates. The ranking uses simple signals such as name match, place importance, and geographic consistency with other places in the same text. It is rule-based, multilingual, and fast, so it is useful as a practical baseline before trying heavier LLM-based methods.

## What you will do

- Understand the principle of UniTopRank.
- Download and install the official repository.
- Reuse any NER tool from Notebook 01: spaCy, Stanza, Flair, or Transformers.
- Choose GeoNames only or UniTopRank's GeoNames+Photon fallback behavior for candidates.
- Rank candidates with UniTopRank.

In [11]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

import importlib.util
import subprocess
import sys
import pandas as pd
from pathlib import Path
from src.config import RESULTS_DIR, GEONAMES_BASE_URL, PHOTON_BASE_URL, REQUEST_TIMEOUT
from src.data_utils import save_dataframe
from src.ner_utils import extract_locations_spacy, extract_locations_stanza, extract_locations_flair, extract_locations_transformers, normalize_ner_results, combine_and_deduplicate_mentions

## Step 1: Method idea

UniTopRank is a fast, CPU-friendly, multilingual toponym resolution method. It is rule-based and interpretable: it ranks candidate places using signals such as name similarity, administrative importance, population, and spatial coherence between places mentioned in the same document.

This makes it useful when you need a strong practical baseline, fast inference, multilingual behavior, or deployment without GPU-heavy LLMs. It is not magic: it still depends on good NER and good candidate retrieval, and very ambiguous texts may need richer context.

## Step 2: Download and install UniTopRank

The official repository is `https://gitlab.com/dlr-dw/UniTopRank`. Run the cell only when UniTopRank is not already installed. After installation, restart the notebook kernel if imports still fail.

In [12]:
RUN_UNITOPRANK_INSTALL = False

unitoprank_dir = PROJECT_ROOT / "external" / "UniTopRank"

if RUN_UNITOPRANK_INSTALL:
    (PROJECT_ROOT / "external").mkdir(exist_ok=True)
    if not unitoprank_dir.exists():
        subprocess.check_call([
            "git",
            "clone",
            "https://gitlab.com/dlr-dw/UniTopRank.git",
            str(unitoprank_dir),
        ])
    requirement_candidates = [
        unitoprank_dir / "requirements.txt",
        unitoprank_dir / "api" / "requirements.txt",
    ]
    requirements_file = next((path for path in requirement_candidates if path.exists()), None)
    if requirements_file is None:
        raise FileNotFoundError(
            "Could not find UniTopRank requirements.txt. Checked: "
            + ", ".join(str(path) for path in requirement_candidates)
        )
    print("Installing UniTopRank requirements from:", requirements_file)
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-r",
        str(requirements_file),
    ])
else:
    print("Skipped UniTopRank install. Set RUN_UNITOPRANK_INSTALL = True if needed.")

Skipped UniTopRank install. Set RUN_UNITOPRANK_INSTALL = True if needed.


## Step 3: Make the UniTopRank API importable

In [13]:
possible_api_dirs = [
    PROJECT_ROOT / "external" / "UniTopRank",
    PROJECT_ROOT / "external" / "UniTopRank" / "api",
    PROJECT_ROOT / "UniTopRank",
    PROJECT_ROOT / "UniTopRank" / "api",
    PROJECT_ROOT / "third_party" / "UniTopRank",
    PROJECT_ROOT / "third_party" / "UniTopRank" / "api",
]

for path in possible_api_dirs:
    if path.exists() and str(path) not in sys.path:
        sys.path.insert(0, str(path))

def add_unitoprank_alias_if_needed():
    # Some repository snapshots contain package folder `unitorank`,
    # while geo_rank_api.py imports `unitoprank`. This alias is only for
    # the current notebook session; it does not edit the external repo.
    if importlib.util.find_spec("unitoprank") is not None:
        return
    if importlib.util.find_spec("unitorank") is None:
        return
    import importlib as importlib_module
    import unitorank
    sys.modules.setdefault("unitoprank", unitorank)
    for name in ["types", "ranker", "candidate_retriever", "ner", "pipeline"]:
        try:
            module = importlib_module.import_module(f"unitorank.{name}")
            sys.modules.setdefault(f"unitoprank.{name}", module)
        except Exception:
            pass

add_unitoprank_alias_if_needed()

try:
    from geo_rank_api import resolve_toponyms
    from geoparsing_api import CandidateRetriever, CandidateRetrieverConfig
    unitoprank_available = True
    unitoprank_import_error = None
except Exception as exc:
    unitoprank_available = False
    unitoprank_import_error = exc

print("geo_rank_api available:", unitoprank_available)

geo_rank_api available: True


## Step 4: Extract toponyms from raw text

Use any NER tool introduced in Notebook 01. The default is spaCy because it is lightweight. If a tool or model is missing, go back to Notebook 01 and run that tool's install cell.

In [14]:
text = "The flood affected Passau and nearby communities along the Danube River in Bavaria."
NER_TOOL = "transformers"  # options: "spacy", "stanza", "flair", "transformers"

def extract_toponyms_with_selected_ner(text, ner_tool="spacy", source_text_id="unitoprank_demo"):
    if ner_tool == "spacy":
        results = extract_locations_spacy(text, model_name="en_core_web_sm")
    elif ner_tool == "stanza":
        results = extract_locations_stanza(text, lang="en")
    elif ner_tool == "flair":
        results = extract_locations_flair(text)
    elif ner_tool == "transformers":
        results = extract_locations_transformers(text)
    else:
        raise ValueError("NER_TOOL must be one of: spacy, stanza, flair, transformers")
    return normalize_ner_results(results, source_text_id=source_text_id)

ner_rows = extract_toponyms_with_selected_ner(text, NER_TOOL)
ner_rows

,mention,start,end,label,tool,source_text_id,sentence
0,Passau,18,25,LOC,transformers,unitoprank_demo,The flood affected Passau and nearby communiti...
1,Danube River,58,71,LOC,transformers,unitoprank_demo,The flood affected Passau and nearby communiti...
2,Bavaria,74,82,LOC,transformers,unitoprank_demo,The flood affected Passau and nearby communiti...


## Step 5: Convert NER output to UniTopRank mentions

In [15]:
def mentions_from_ner(df):
    mentions = []
    for _, row in df.dropna(subset=["mention"]).iterrows():
        if pd.isna(row.get("start")) or pd.isna(row.get("end")):
            continue
        mentions.append({
            "LOC": str(row["mention"]),
            "start": int(row["start"]),
            "end": int(row["end"]),
        })
    return mentions

mentions = mentions_from_ner(ner_rows)
mentions

[{'LOC': 'Passau', 'start': 18, 'end': 25},
 {'LOC': 'Danube River', 'start': 58, 'end': 71},
 {'LOC': 'Bavaria', 'start': 74, 'end': 82}]

## Step 6: Choose candidate source

UniTopRank ranks candidates; it does not create coordinates from nothing. The official repository uses `CandidateRetriever` with `CandidateRetrieverConfig` before calling `resolve_toponyms(...)`.

Use one of two modes:

- `geonames`: GeoNames only.
- `geonames_photon`: UniTopRank's default two-geocoder strategy. GeoNames is tried first. Photon is used only when GeoNames returns no candidates or no high-quality name match. This follows `photon_enabled=True` with `bool_merge=False` in the UniTopRank repository.

There is also a `bool_merge=True` option in the repository that always merges Photon with GeoNames, but this notebook keeps the default UniTopRank behavior.

In [16]:
CANDIDATE_SOURCE = "geonames_photon"  # options: "geonames", "geonames_photon"

def unitoprank_geonames_prefix(base_url):
    base = str(base_url).strip()
    if "location=" in base:
        return base
    separator = "&" if "?" in base else "?"
    return f"{base}{separator}location="

def unitoprank_photon_prefix(base_url):
    base = str(base_url).strip()
    if "q=" in base:
        return base
    separator = "&" if "?" in base else "?"
    return f"{base}{separator}q="

use_photon = CANDIDATE_SOURCE == "geonames_photon"
force_photon_merge = False  # keep UniTopRank default behavior

if not unitoprank_available:
    raise ImportError(
        "UniTopRank is not available. Run the install cell in Step 2, then rerun Step 3. "
        f"Import error was: {unitoprank_import_error!r}"
    )

def make_candidate_retriever(candidate_source):
    use_photon = candidate_source == "geonames_photon"
    if candidate_source not in ("geonames", "geonames_photon"):
        raise ValueError("candidate_source must be 'geonames' or 'geonames_photon'")
    return CandidateRetriever(
        CandidateRetrieverConfig(
            geonames_enabled=True,
            photon_enabled=use_photon,
            bool_merge=False,  # UniTopRank default: Photon is fallback/supplement, not always merged.
            geonames_url=unitoprank_geonames_prefix(GEONAMES_BASE_URL),
            photon_url=unitoprank_photon_prefix(PHOTON_BASE_URL),
            geonames_limit=40,
            photon_limit=300,
            timeout_seconds=int(REQUEST_TIMEOUT),
            deduplicate_by_address=False,
        )
    )

retriever = make_candidate_retriever(CANDIDATE_SOURCE)

print("Candidate source:", CANDIDATE_SOURCE)
print("Photon enabled:", retriever.config.photon_enabled)
print("Force Photon merge:", retriever.config.bool_merge)
print("GeoNames URL prefix:", retriever.config.geonames_url)
print("Photon URL prefix:", retriever.config.photon_url if use_photon else "(disabled)")

Candidate source: geonames_photon
Photon enabled: True
Force Photon merge: False
GeoNames URL prefix: http://dw-mir-postgis.intra.dlr.de:8091/location?location=
Photon URL prefix: http://photon.intra.dlr.de:2322/api/?q=


## Step 7: Retrieve candidates with UniTopRank's retriever

This step uses only the configured GeoNames/Photon services. If no candidates are returned, check the service URLs and DLR network/VPN access before continuing.

## Step 8: Rank with UniTopRank

This is the core method call. If UniTopRank is not installed yet, install it in Step 2 and rerun from Step 3.

In [17]:
def retrieve_candidates_for_mentions(mentions, retriever):
    names = [m["LOC"] for m in mentions]
    if not names:
        print("No toponyms available for candidate retrieval.")
        return {}

    try:
        candidates_by_toponym = retriever.get_candidates_for_toponyms(names)
    except Exception as exc:
        print("Candidate retrieval failed. Check GeoNames/Photon service URLs and DLR network/VPN access.")
        print("Error:", repr(exc))
        return {name.casefold(): [] for name in names}

    if not any(bool(candidates) for candidates in candidates_by_toponym.values()):
        print("No candidates returned by GeoNames/Photon. Check service URLs, network/VPN access, and the selected candidate source.")
    return candidates_by_toponym


def preview_candidates(candidates_by_toponym):
    rows = []
    for query, candidates in candidates_by_toponym.items():
        for rank, cand in enumerate(candidates, start=1):
            rows.append({
                "query": query,
                "rank": rank,
                "name": cand.get("name"),
                "address": cand.get("address"),
                "lat": cand.get("lat"),
                "lon": cand.get("lon"),
                "population": cand.get("population"),
                "admin_level": cand.get("admin_level"),
            })
    return pd.DataFrame(rows)

candidates_by_toponym = retrieve_candidates_for_mentions(mentions, retriever)
preview_candidates(candidates_by_toponym).head(50)

,query,rank,name,address,lat,lon,population,admin_level
0,passau,1,Passau,"Passau, Lower Bavaria, Bavaria, Germany, Europe",48.57318,13.45060,53093,ADM4
1,passau,2,Passau,"Passau, Politischer Bezirk Hermagor, Carinthia...",46.68207,12.95052,2,PPL
2,passau,3,Passau,"Passau, Kabupaten Majene, West Sulawesi, Indon...",-3.39300,118.86840,0,PPL
3,passau,4,Passau,"Passau, Politischer Bezirk Sankt Johann im Pon...",47.08333,13.11667,0,PPL
4,passau,5,Passau,"Passau, Schleswig-Holstein, Germany, Europe",54.26409,10.30913,0,STM
5,passau,6,Landkreis Passau,"Landkreis Passau, Lower Bavaria, Bavaria, Germ...",48.59944,13.29389,194090,ADM3
6,passau,7,Kreisfreie Stadt Passau,"Kreisfreie Stadt Passau, Lower Bavaria, Bavari...",48.58194,13.43306,53093,ADM3
7,passau,8,Passau,"Passau, Lower Bavaria, Bavaria, Germany, Europe",48.56650,13.43122,50560,PPLA3
8,passau,9,Passau-Vilshofen,"Passau-Vilshofen, Lower Bavaria, Bavaria, Germ...",48.63333,13.20000,0,AIRF
9,passau,10,"Hofkirchen, Kr. Passau, Pfarrkirche Mariä Himm...","Hofkirchen, Kr. Passau, Pfarrkirche Mariä Himm...",48.67812,13.11725,0,CH


In [18]:
if not unitoprank_available:
    raise ImportError(
        "UniTopRank is not available. Run the install cell in Step 2, then rerun Step 3. "
        f"Import error was: {unitoprank_import_error!r}"
    )

if not any(bool(candidates) for candidates in candidates_by_toponym.values()):
    print("Skipping UniTopRank ranking because no candidates were retrieved.")
    resolved_mentions = []
    ranked_results = {}
else:
    resolved_mentions, ranked_results = resolve_toponyms(
        text=text,
        mentions=mentions,
        candidates_by_toponym=candidates_by_toponym,
        top_n=10,
    )

resolved_mentions

[{'LOC': 'Passau',
  'start': 18,
  'end': 25,
  'lat': 48.57318,
  'lon': 13.4506,
  'address': 'Passau, Lower Bavaria, Bavaria, Germany, Europe'},
 {'LOC': 'Danube River',
  'start': 58,
  'end': 71,
  'lat': 45.33333,
  'lon': 29.66667,
  'address': 'Danube River, Romania, Europe'},
 {'LOC': 'Bavaria',
  'start': 74,
  'end': 82,
  'lat': 49.0,
  'lon': 11.5,
  'address': 'Bavaria, Germany, Europe'}]

## Step 9: Save the selected locations

In [19]:
def normalize_unitoprank_results(resolved_mentions, candidate_source):
    rows = []
    for item in resolved_mentions:
        rows.append({
            "mention": item.get("LOC") or item.get("text") or item.get("mention"),
            "selected_name": item.get("name") or item.get("address"),
            "lat": item.get("lat"),
            "lon": item.get("lon"),
            "method": f"unitoprank_{candidate_source}",
        })
    return pd.DataFrame(rows)


results = normalize_unitoprank_results(resolved_mentions, CANDIDATE_SOURCE)
out = save_dataframe(results, RESULTS_DIR / "unitoprank_results.csv")
print("Saved:", out)
results

Saved: /home/hu_xk/Workplace/wawopensearch3_hackathon/module2_geoparsing/outputs/results/unitoprank_results.csv


,mention,selected_name,lat,lon,method
0,Passau,"Passau, Lower Bavaria, Bavaria, Germany, Europe",48.57318,13.45060,unitoprank_geonames_photon
1,Danube River,"Danube River, Romania, Europe",45.33333,29.66667,unitoprank_geonames_photon
2,Bavaria,"Bavaria, Germany, Europe",49.00000,11.50000,unitoprank_geonames_photon


## Step 10: Play with your own text


In [10]:
#my_text = "My institute locates in Jena, which is in Louisiana, the US."
my_text = "My institute locates in Jena, which is in Germany."

MY_NER_TOOL = "transformers"  # options: "spacy", "stanza", "flair", "transformers"
MY_CANDIDATE_SOURCE = "geonames_photon"  # options: "geonames", "geonames_photon"

my_ner = extract_toponyms_with_selected_ner(
    my_text,
    ner_tool=MY_NER_TOOL,
    source_text_id="my_unitoprank_text",
)
my_mentions = mentions_from_ner(my_ner)

my_retriever = make_candidate_retriever(MY_CANDIDATE_SOURCE)
my_candidates_by_toponym = retrieve_candidates_for_mentions(my_mentions, my_retriever)

display(preview_candidates(my_candidates_by_toponym).head(30))

if not any(bool(candidates) for candidates in my_candidates_by_toponym.values()):
    print("Skipping UniTopRank ranking because no candidates were retrieved for your text.")
    my_resolved = []
    my_ranked = {}
else:
    my_resolved, my_ranked = resolve_toponyms(
        text=my_text,
        mentions=my_mentions,
        candidates_by_toponym=my_candidates_by_toponym,
        top_n=10,
    )

my_resolved

No candidates returned by GeoNames/Photon. Check service URLs, network/VPN access, and the selected candidate source.


""


Skipping UniTopRank ranking because no candidates were retrieved for your text.


[]

## Discussion

UniTopRank is strong when candidate lists are reasonable and the document has useful geographic context. It is often attractive for multilingual, CPU-only, large-scale, or near-real-time workflows.

It can struggle when NER misses a place, candidate retrieval returns poor candidates, the text is very short, or the correct place requires external world knowledge not present in the text.

## Reference

- UniTopRank repository: https://gitlab.com/dlr-dw/UniTopRank
- Hu, X., Sun, Y., Hecking, T., Kersten, J., & Klan, F. (2026). *UniTopRank: a scalable and language-independent method for toponym resolution*. International Journal of Geographical Information Science. https://doi.org/10.1080/13658816.2026.2645831